In [7]:
import pygsheets # use 'pip install pygsheets', not maintained with conda

import pandas
import datetime
import numpy

# terminals

In [8]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
spreadsheet = gc.open_by_key('1tcS6Wd-Wp-LTDpLzFgJY_RSNDnbyubW3J_9HKIAys4A')
#spreadsheet = gc.open_by_key('1MrghwBeCz8Tzgua7CWGg_KoXKVZsV7r0kHMYHYqnNTg') # for July 2022 version update (terminals)

terms_df_orig = spreadsheet.worksheet('title', 'Terminals').get_as_df(start='A2')
terms_dict_df = spreadsheet.worksheet('title', 'Data dictionary').get_as_df()
terms_acronyms_df = spreadsheet.worksheet('title', 'Acronyms').get_as_df()
terms_copyright_df = spreadsheet.worksheet('title', 'Copyright').get_as_df()

/Users/baird/miniconda3/envs/gem/lib/python3.9/site-packages/pygsheets/worksheet.py:1477: UserWarning: At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.
  warnings.warn('At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.')


## clean up terminals

In [9]:
# remove oil export terminals
terms_df_orig = terms_df_orig[terms_df_orig['Fuel']=='LNG']
# remove anything without a wiki page
terms_df_orig = terms_df_orig[terms_df_orig['Wiki']!='']

## subset columns to include relevant columns

Note these columns are specified in the "Data dictionary" tab of the Pipelines_Current sheet; the column IncludeWithDataRelease has a Yes if the column is included, and DataReleaseColumnOrder determines how they're arranged.

In [10]:
terms_dict_df_include = terms_dict_df.copy()[terms_dict_df.copy()['IncludeWithDataRelease']=='Yes']
terms_dict_df_include = terms_dict_df_include.sort_values('DataReleaseColumnOrder', ascending=True)
terms_dict_df_include = terms_dict_df_include[['VariableName','Definition']]

In [11]:
terms_df_subset = terms_df_orig.copy()[terms_dict_df_include['VariableName'].tolist()]
terms_df_subset.sort_values('ComboID', inplace=True)

## write to Excel file

Use ExcelFormatter from Pandas to write 4 different tabs to the same file.

In [13]:
#pandas.io.formats.excel.ExcelFormatter.header_style = None

excel_writer = pandas.ExcelWriter('GEM-LNG-Terminals-'+str(datetime.date.today())+'.xlsx', engine='xlsxwriter')

terms_df_subset.to_excel(excel_writer, sheet_name='LNG Terminals '+str(datetime.date.today()), index=False)
terms_dict_df_include.to_excel(excel_writer, sheet_name='Data dictionary', index=False)
terms_acronyms_df.to_excel(excel_writer, sheet_name='Acronyms', index=False)
terms_copyright_df.to_excel(excel_writer, sheet_name='Copyright', index=False)

excel_writer.save()

/var/folders/h5/nfk59_vx7gg_58myc9jfk7_40000gn/T/ipykernel_6428/938021473.py:10: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  excel_writer.save()


# pipelines

In [20]:
#fuel_type = 'Gas'
fuel_type = 'Oil'
#fuel_type = 'Oil-and-Gas'

credentials_directory = '/Users/baird/Dropbox/_google-api/'
gc = pygsheets.authorize(client_secret=credentials_directory+'client_secret.json')

spreadsheet = gc.open_by_key('1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek')
#spreadsheet[1] "Gas Pipelines" tab is the second index
gas_pipes = spreadsheet.worksheet('title', 'Gas pipelines').get_as_df(start='A2')
oil_pipes = spreadsheet.worksheet('title', 'Oil/NGL pipelines').get_as_df(start='A2')
#owners = spreadsheet[3].get_as_df()

#remove liquid pipelines you don't want to distribute
remove_fuels = ['Carbon']
oil_pipes = oil_pipes[~oil_pipes['Fuel'].isin(remove_fuels)]

pipes_dict_df = spreadsheet.worksheet('title', 'Data dictionary').get_as_df()
pipes_acronyms_df = spreadsheet.worksheet('title', 'Acronyms').get_as_df()

if fuel_type == 'Gas':
    pipes_df_orig = gas_pipes.copy() #pandas.concat([oil_pipes, gas_pipes], ignore_index=True)
if fuel_type == 'Oil':
    pipes_df_orig = oil_pipes.copy()
if fuel_type == 'Oil-and-Gas':
    pipes_df_orig = pandas.concat([oil_pipes, gas_pipes], ignore_index=True)

if fuel_type == 'Gas':
    pipes_copyright_df = spreadsheet.worksheet('title', 'Copyright - GGIT').get_as_df()
elif fuel_type == 'Oil':
    pipes_copyright_df = spreadsheet.worksheet('title', 'Copyright - GOIT').get_as_df()

/Users/baird/miniconda3/envs/gem/lib/python3.9/site-packages/pygsheets/worksheet.py:1477: UserWarning: At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.
  warnings.warn('At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.')


## clean up pipelines

In [21]:
# clean up rows that should not be distributed
pipes_df_orig = pipes_df_orig[pipes_df_orig['Status']!='N/A']
pipes_df_orig = pipes_df_orig[pipes_df_orig['PipelineName']!='']

In [22]:
status_in_dev = ['Proposed', 
                 'Construction', 
                 'Shelved', 'Operating', 
                 'Mothballed', 
                 'Cancelled', 
                 'Retired', 
                 'Idle']
no_route_options = [
    'Unavailable', 
    'Capacity expansion only', 
    'Bidirectionality upgrade only',
    'Short route (< 100 km)', 
    'N/A',
    ''
]

# filter for the statuses above in the status_in_dev list (modify as desired)
#gas_pipes = gas_pipes[gas_pipes['Status'].str.lower().isin(status_in_dev)]

## subset pipeline columns to include relevant columns

Note these columns are specified in the "Data dictionary" tab of the Pipelines_Current sheet; the column IncludeWithDataRelease has a Yes if the column is included, and DataReleaseColumnOrder determines how they're arranged.

In [23]:
pipes_dict_df_include = pipes_dict_df.copy()[(pipes_dict_df.copy()['IncludeWithDataRelease']=='Yes')&
                                             (pipes_dict_df.copy()['GasVariable']=='Yes')]
pipes_dict_df_include = pipes_dict_df_include.sort_values('DataReleaseColumnOrder', ascending=True)
pipes_dict_df_include = pipes_dict_df_include[['VariableName','Definition']]

In [24]:
pipes_df_subset = pipes_df_orig.copy()[pipes_dict_df_include['VariableName'].tolist()]

## write to Excel file

Use ExcelFormatter from Pandas to write 4 different tabs to the same file.

In [25]:
#pandas.io.formats.excel.ExcelFormatter.header_style = None

excel_writer = pandas.ExcelWriter('GEM-'+fuel_type+'-Pipelines-'+str(datetime.date.today())+'.xlsx', engine='xlsxwriter')

pipes_df_subset.to_excel(excel_writer, sheet_name='Pipelines '+str(datetime.date.today()), index=False)
pipes_dict_df_include.to_excel(excel_writer, sheet_name='Data dictionary', index=False)
pipes_acronyms_df.to_excel(excel_writer, sheet_name='Acronyms', index=False)
if fuel_type in ['Oil','Gas']:
    pipes_copyright_df.to_excel(excel_writer, sheet_name='Copyright', index=False)

#workbook = excel_writer.book
#worksheet = excel_writer.sheets['Terminals '+str(datetime.date.today())]
#format = workbook.add_format({'text_wrap': True})

excel_writer.save()

/var/folders/h5/nfk59_vx7gg_58myc9jfk7_40000gn/T/ipykernel_6428/2099933109.py:15: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  excel_writer.save()
